In [1]:
import json
import random

# The strict system prompt the model must obey
SYSTEM_PROMPT = "You are an on-device mobile assistant. If the user request matches a tool, emit ONLY a JSON object wrapped in <tool_call>...</tool_call>. If no tool matches, reply with a plain text refusal."

def make_convo(user_text, assistant_text):
    return {"messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text}
    ]}

data = [
    # 1. Weather Tool
    make_convo("What's the weather in Tokyo?", '<tool_call>{"tool": "weather", "args": {"location": "Tokyo", "unit": "C"}}</tool_call>'),
    make_convo("how hot is it in miami in fahrenheit", '<tool_call>{"tool": "weather", "args": {"location": "Miami", "unit": "F"}}</tool_call>'),

    # 2. Calendar Tool
    make_convo("List my events for tomorrow", '<tool_call>{"tool": "calendar", "args": {"action": "list", "date": "2026-04-19"}}</tool_call>'),
    make_convo("Create a meeting for 2026-04-20 titled Project Sync", '<tool_call>{"tool": "calendar", "args": {"action": "create", "date": "2026-04-20", "title": "Project Sync"}}</tool_call>'),

    # 3. Convert Tool
    make_convo("Convert 50 kg to lbs", '<tool_call>{"tool": "convert", "args": {"value": 50, "from_unit": "kg", "to_unit": "lbs"}}</tool_call>'),
    make_convo("100 miles in kilometers plz", '<tool_call>{"tool": "convert", "args": {"value": 100, "from_unit": "miles", "to_unit": "kilometers"}}</tool_call>'),

    # 4. Currency Tool
    make_convo("Change 500 USD to EUR", '<tool_call>{"tool": "currency", "args": {"amount": 500, "from": "USD", "to": "EUR"}}</tool_call>'),
    make_convo("mujhe 1000 PKR to USD batao", '<tool_call>{"tool": "currency", "args": {"amount": 1000, "from": "PKR", "to": "USD"}}</tool_call>'), # Adversarial code-switch

    # 5. SQL Tool
    make_convo("Get all users where age > 18", '<tool_call>{"tool": "sql", "args": {"query": "SELECT * FROM users WHERE age > 18"}}</tool_call>'),

    # 6. Refusals (Crucial for Slice D)
    make_convo("Play some music by Atif Aslam", "I'm sorry, I don't have a tool to play music."),
    make_convo("Turn on the living room lights.", "I cannot control smart home devices."),
    make_convo("Bhai ek pizza order kar do", "I don't have the ability to place food orders."),

    # 7. Adversarial Typos (Crucial for Slice C)
    make_convo("wethr in lndon in C", '<tool_call>{"tool": "weather", "args": {"location": "London", "unit": "C"}}</tool_call>'),
    make_convo("cnvrt 10 cm 2 inches", '<tool_call>{"tool": "convert", "args": {"value": 10, "from_unit": "cm", "to_unit": "inches"}}</tool_call>')
]

# Duplicate and shuffle to create a decent sized training batch
training_data = data * 5
random.shuffle(training_data)

with open('synthetic_train.jsonl', 'w') as f:
    for item in training_data:
        f.write(json.dumps(item) + '\n')

print(f"Generated {len(training_data)} training examples and saved to synthetic_train.jsonl!")

Generated 70 training examples and saved to synthetic_train.jsonl!


In [2]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets

In [3]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Load the tiny, fast base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 2. Add LoRA adapters for fast training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 3. Load our synthetic data
dataset = load_dataset("json", data_files="synthetic_train.jsonl", split="train")

def format_data(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_data, batched = True)

# 4. Execute the training run
print("Starting Training...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Fast burst training
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        output_dir = "outputs",
    ),
)
trainer.train()

# 5. The "Antigravity" Export: Save as a compressed GGUF
print("Exporting compressed model for CPU inference...")
model.save_pretrained_gguf("pocket_agent", tokenizer, quantization_method = "q4_k_m")
print("DONE! Check the Colab files sidebar for your .gguf file.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:09<00:00,  9.51s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:10<00:00, 10.64s/it]


Unsloth: Merge process complete. Saved to `/content/pocket_agent`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['pocket_agent_gguf/qwen2.5-0.5b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['pocket_agent_gguf/qwen2.5-0.5b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model pocket_agent_gguf/qwen2.5-0.5b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to pocket_agent_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f pocket_agent_gguf/Modelfile
DONE! Check the Colab files sidebar for your .gguf file.


In [4]:
import shutil

# 1. This extracts just the lightweight LoRA weights from the model in memory
print("Extracting LoRA adapter...")
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

# 2. This zips it up into one file so you can actually download it
print("Zipping the folder...")
shutil.make_archive('lora_adapter', 'zip', 'lora_adapter')

print("✅ DONE! Look in the file browser for 'lora_adapter.zip'")

Extracting LoRA adapter...
Zipping the folder...
✅ DONE! Look in the file browser for 'lora_adapter.zip'
